In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.nonparametric.smoothers_lowess import lowess
import math
import pandas as pd
import os
from scipy.interpolate import splrep, splev


def interplote_data(list, num):
    x = np.linspace(0,1, len(list))
    spl = splrep(x, list)
    new_x = np.linspace(0,1, num)
    new_value = splev(new_x, spl)
    return new_value

def find_baseline(data):
    n = len(data)

    start_index = int(n * 0.30)
    end_index = int(n * 0.70)
    middle_40_percent = data[start_index:end_index]
    min_value = min(middle_40_percent)
    min_indices_relative = [i for i, value in enumerate(middle_40_percent) if value == min_value]
    min_indices_absolute = start_index + min_indices_relative[0]

    baseline_value = data[min_indices_absolute-int(n*0.005):min_indices_absolute+int(n*0.005)]
    baseline_average = np.mean(baseline_value)
    return baseline_average

def find_crops(data, target_value):
    data_as_int = [int(float(x)) for x in data]
    matching_indices = [i for i, value in enumerate(data_as_int) if value<target_value]
    mini_crop = 0
    max_crop = len(data)
    if 0 in matching_indices:
        i = 0
        while i in range(0,len(matching_indices)):
            if 0<(matching_indices[i+1]-matching_indices[i]) <=2:
                mini_crop = matching_indices[i+1] 
                i += 1
            else:
                break

    reverse_matching_indices = matching_indices[::-1]

    if (len(data)-1) in reverse_matching_indices:
        i = 0
        while i in range(0,len(reverse_matching_indices)):
            if 0<(reverse_matching_indices[i]-reverse_matching_indices[i+1]) <=2:
                max_crop = reverse_matching_indices[i+1] 
                i += 1
            else:
                break

    return mini_crop, max_crop

def lowess_smooth (data):
    x = np.linspace(0, 1, len(data))
    y = data

    frac = 0.05 # IMPORTANT window length
    lowess_result = lowess(y, x, frac=frac)

    return lowess_result[:,1]

def normalize_data(list):
    list_max = max(list)
    normalized_data_1d = [ (data / list_max)for data in list]
    return normalized_data_1d

def interpolate_data_b(x, y, num):
    spl = splrep(x, y)  
    new_x = np.linspace(min(x), max(x), num)  
    new_value = splev(new_x, spl)
    return new_value

def half_data (list, num):
    num_r = math.ceil(num/2)
    front_half = list[:num_r]
    back_half = list[num_r:]
    front_half_reversed = front_half[::-1]
    
    return front_half_reversed,back_half

def ave_center_data(data):
    n = len(data)

    start = int(n * 0.495)
    end = int(n * 0.595)
    middle_1_percent = data[start:end]

    baseline = sum(middle_1_percent) / len(middle_1_percent) 
    
    return baseline

def fraction (list):
    total_sum = sum(list)
    cumulative_sum = []
    current_sum = 0
    for value in list:
        current_sum += value
        cumulative_sum.append(current_sum)
    cumulative_weights = [(cs / total_sum)for cs in cumulative_sum]
    return cumulative_weights

def fig_4_create (new_red,new_green):
    
    # Green
    front_green,back_green = half_data (new_green, len(new_green))

    x_front = fraction (front_green)

    x_back = fraction (back_green)

    # Red

    baseline = ave_center_data (new_red)
    front_red,back_red = half_data (new_red, len(new_red))

    front_red_b = [(x/baseline) - 1 for x in front_red]
    back_red_b = [(x/baseline) - 1 for x in back_red]

    intensity_front = interpolate_data_b(x_front, front_red_b, 1000)
    intensity_back = interpolate_data_b(x_back, back_red_b, 1000)

    return intensity_front, intensity_back

def calculate_plot_parameter(intensities):
    average = np.mean(intensities, axis=0)
    average = lowess_smooth(average)

    stderr = np.std(intensities, axis=0) / np.sqrt(len(intensities))
    stderr = lowess_smooth(stderr)

    std = np.std(intensities, axis=0)
    std = lowess_smooth(std)

    return average, stderr, std

def adjust_the_line(red_cell, green_cell):

    # Red
    new_red_value = interplote_data(red_cell, 1000)

    average = lowess_smooth(new_red_value)

    baseline_average = find_baseline(average)
    mini_crop, max_crop = find_crops(average, baseline_average)
    data_new = average[(mini_crop): (max_crop)]
    # Perform interpolation
    x = np.linspace(0,1, len(data_new))
    y = data_new
    spl = splrep(x, y)  # Fit the spline
    new_x = np.linspace(x.min(), x.max(), 1000)  
    new_red_y = splev(new_x, spl) 

    # Green
    new_green_value = interplote_data(green_cell, 1000)
    
    smooth_green = lowess_smooth(new_green_value)
    data_new_green = smooth_green[(mini_crop): (max_crop)]
    # Perform interpolation
    x = np.linspace(0,1, len(data_new_green))
    y = data_new_green
    spl = splrep(x, y)  # Fit the spline
    new_x = np.linspace(x.min(), x.max(), 1000)  
    new_green_y = splev(new_x, spl) 

    
    return new_red_y,new_green_y

def figure_3_create(new_red,new_green):
    # Green
    front_green,back_green = half_data (new_green, len(new_green))
    x_front = fraction (front_green)
    x_back = fraction (back_green)

    # Red
    front_red,back_red = half_data (new_red, len(new_red))

    #Integrate
    intensity_front = interpolate_data_b (x_front, front_red, 1000)
    intensity_front = normalize_data(intensity_front)

    intensity_back = interpolate_data_b (x_back, back_red, 1000)
    intensity_back = normalize_data(intensity_back)

    return intensity_front, intensity_back


def allinfunction(folder_path):
    csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]


    red_values_list = []
    green_values_list = []
    fig_3_intensities = []
    fig_4_intensities = []
    fig_5_list = []

    for file in csv_files: # range(0,len(df['Red_intensity']))
        try:
            df = pd.read_csv(os.path.join(folder_path, file))
            red_cell = df.iloc[:, 2].tolist() 
            green_cell = df.iloc[:, 1].tolist() 
            # new_red,new_green = adjust_the_line (red_cell, green_cell)
            new_green_1, new_red_1 = adjust_the_line (green_cell, red_cell)

            new_green = [(i-100) for i in new_green_1]
            new_red = [(i-100) for i in new_red_1]

            # This is figure 1 & 2 parts
            normalized_red_value = normalize_data(new_red)
            normalized_green_value = normalize_data(new_green)
            red_values_list.append(normalized_red_value)
            green_values_list.append(normalized_green_value)

            # This is figure 3 part
            intensity_front_3, intensity_back_3 = figure_3_create(new_red,new_green)
            fig_3_intensities.append(intensity_front_3)
            fig_3_intensities.append(intensity_back_3)

            # This is figure 4 part
            intensity_front_4, intensity_back_4 = fig_4_create (new_red,new_green)
            averaged_list = [(a + b) / 2 for a, b in zip(intensity_front_4, intensity_back_4)]
            fig_4_intensities.append(averaged_list)

        except:
            continue

    return red_values_list, green_values_list, fig_3_intensities, fig_4_intensities



# concentration

## 50nM

In [ ]:
file_path = r"50nM rapa data all"

red_values_list, green_values_list, fig_3_intensities, fig_4_intensities_50 = allinfunction(file_path)

average_50, stderr_50, std_50 = calculate_plot_parameter(fig_4_intensities_50)


## 25nM

In [ ]:
file_path = r"25nM rapa data all"

red_values_list, green_values_list, fig_3_intensities, fig_4_intensities_25 = allinfunction(file_path)

average_25, stderr_25, std_25 = calculate_plot_parameter(fig_4_intensities_25)

## control

In [ ]:
file_path = r"control data all"

red_values_list, green_values_list, fig_3_intensities, fig_4_intensities_0 = allinfunction(file_path)

average_0, stderr_0, std_0 = calculate_plot_parameter(fig_4_intensities_0)

In [ ]:
import matplotlib.pyplot as plt


fig, ax = plt.subplots(figsize=(4.5, 3.2), dpi=300)

# 50 nM rapamycin
ax.plot(
    average_50,
    color="#128ACB",
    linewidth=2.5,
    label="50 nM Rapa"
)
ax.fill_between(
    range(len(average_50)),
    average_50 - stderr_50,
    average_50 + stderr_50,
    color="#128ACB",
    alpha=0.25,
    linewidth=0
)

# 25 nM rapamycin
ax.plot(
    average_25,
    color="#59BDF3",
    linewidth=2.5,
    label="25 nM Rapa"
)
ax.fill_between(
    range(len(average_25)),
    average_25 - stderr_25,
    average_25 + stderr_25,
    color="#59BDF3",
    alpha=0.25,
    linewidth=0
)

# Control
ax.plot(
    average_0,
    color="#515050",
    linewidth=2.5,
    label="Control"
)
ax.fill_between(
    range(len(average_0)),
    average_0 - stderr_0,
    average_0 + stderr_0,
    color="gray",
    alpha=0.25,
    linewidth=0
)

# Reference line
ax.axhline(
    y=0,
    color="gray",
    linestyle="--",
    linewidth=1,
    zorder=0
)

# Axes
ax.set_xticks(
    [0, len(average_0) // 2, len(average_0) - 1],
    [0, 0.5, 1]
)

ax.set_yticks([0, 0.5, 1, 1.5])

ax.set_xlim(0, len(average_0) - 1)
ax.set_ylim(-0.25, 1.5)

ax.set_xlabel(
    "Fraction of intensity line integral from center"
)
ax.set_ylabel(
    "signal monomer intensity change from baseline"
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(
    loc="upper left",
    prop={
        "family": "Arial",
        "size": 12
    },
    frameon=False
)

# Apply layout before saving
fig.tight_layout()

plt.show()

# duration

## control

In [ ]:
file_path = r"control data all"

red_values_list, green_values_list, fig_3_intensities, fig_4_intensities_control_5 = allinfunction(file_path)
average_0, stderr_0, std_0 = calculate_plot_parameter(fig_4_intensities_control_5)

## 12h

In [ ]:
file_path = r"rapa 12h data all"

red_values_list, green_values_list, fig_3_intensities, fig_4_intensities_12 = allinfunction(file_path)

average_12, stderr_12, std_12 = calculate_plot_parameter(fig_4_intensities_12)

## 6h

In [ ]:
file_path = r"rapa 6h data all"

red_values_list, green_values_list, fig_3_intensities, fig_4_intensities_6 = allinfunction(file_path)

average_6, stderr_6, std_6 = calculate_plot_parameter(fig_4_intensities_6)

## 3h

In [ ]:
file_path = r"rapa 3h data all"
red_values_list, green_values_list, fig_3_intensities, fig_4_intensities_3 = allinfunction(file_path)

average_3, stderr_3, std_3 = calculate_plot_parameter(fig_4_intensities_3)

In [ ]:
import matplotlib.pyplot as plt


fig, ax = plt.subplots(figsize=(4.5, 3.2), dpi=300)

# 12 h
ax.plot(
    average_12,
    color="#016499",
    linewidth=2.5,
    label="12 hours"
)
ax.fill_between(
    range(len(average_12)),
    average_12 - stderr_12,
    average_12 + stderr_12,
    color="#016499",
    alpha=0.25,
    linewidth=0
)

# 6 h
ax.plot(
    average_6,
    color="#128ACB",
    linewidth=2.5,
    label="6 hours"
)
ax.fill_between(
    range(len(average_6)),
    average_6 - stderr_6,
    average_6 + stderr_6,
    color="#128ACB",
    alpha=0.25,
    linewidth=0
)

# 3 h
ax.plot(
    average_3,
    color="#59BDF3",
    linewidth=2.5,
    label="3 hours"
)
ax.fill_between(
    range(len(average_3)),
    average_3 - stderr_3,
    average_3 + stderr_3,
    color="#59BDF3",
    alpha=0.25,
    linewidth=0
)

# Control
ax.plot(
    average_0,
    color="#515050",
    linewidth=2.5,
    label="Control"
)
ax.fill_between(
    range(len(average_0)),
    average_0 - stderr_0,
    average_0 + stderr_0,
    color="gray",
    alpha=0.25,
    linewidth=0
)

# Horizontal reference line
ax.axhline(
    y=0,
    color="gray",
    linestyle="--",
    linewidth=1,
    zorder=0
)

# Axes
ax.set_xticks(
    [0, len(average_0) // 2, len(average_0) - 1],
    [0, 0.5, 1]
)

ax.set_yticks([0, 0.5, 1, 1.5])

ax.set_xlim(0, len(average_0) - 1)
ax.set_ylim(-0.25, 1.5)

ax.set_xlabel(
    "Fraction of intensity line integral from center"
)
ax.set_ylabel(
    "signal monomer intensity change from baseline"
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(
    loc="upper left",
    prop={
        "family": "Arial",
        "size": 12
    },
    frameon=False
)


fig.tight_layout()

plt.show()